# 03 · Modelo TF-IDF

**Entrada:** `data/processed/recetas_modelo.csv` (salida del notebook 02).
**Salida:** `models/recetas_modelo/tfidf_vectorizer.joblib` y `recipe_tfidf_matrix.joblib`, los dos archivos que carga el backend.

El modelo lee **una sola columna**, `recipe_text` (nombre + ingredientes base + etiquetas + país). TF-IDF le da a cada palabra un peso dentro de cada receta: sube si aparece en ella y baja si aparece en muchas recetas. Dos recetas se parecen si sus vectores apuntan en la misma dirección (similitud coseno).

| Ajuste | Por qué |
|---|---|
| `ngram_range=(1, 1)` | Solo palabras sueltas. Con pares de palabras salían combinaciones de dos ingredientes distintos (`lechuga frijol`) que casi nunca se repiten y recibían demasiado peso. |
| `min_df=2` | Ignora palabras que salen en una sola receta (erratas). |
| `max_df=0.5` | Ignora palabras en más de la mitad de las recetas (`sal`): no distinguen un plato de otro. |
| `sublinear_tf=True` | Repetir `pollo` cinco veces no hace ganar a una receta. |

In [ ]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))                       # para importar src/

DATA_PATH = ROOT / "data" / "processed" / "recetas_modelo.csv"
MODELS_DIR = ROOT / "models" / "recetas_modelo"     # la carpeta que lee la app (src/config.py)

df = pd.read_csv(DATA_PATH)
assert df["recipe_text"].notna().all() and (df["recipe_text"].str.strip() != "").all(), "hay textos vacíos"
assert df["recipe_id"].is_unique, "hay recetas repetidas"
print(f"Recetas: {len(df):,}")

## 1. Entrenar

In [ ]:
TFIDF_PARAMS = dict(lowercase=True, strip_accents="unicode", ngram_range=(1, 1),
                    min_df=2, max_df=0.5, sublinear_tf=True)

tfidf = TfidfVectorizer(**TFIDF_PARAMS)
tfidf_matrix = tfidf.fit_transform(df["recipe_text"])      # una fila por receta, mismo orden que el CSV
print(f"Matriz: {tfidf_matrix.shape[0]:,} recetas x {tfidf_matrix.shape[1]:,} palabras")

## 2. Probar: "con lo que tengo" y "recetas parecidas"

In [ ]:
def recomendar_por_texto(texto, n=8):
    """Las n recetas más parecidas a un texto escrito (por ejemplo, una lista de ingredientes)."""
    similitudes = cosine_similarity(tfidf.transform([texto]), tfidf_matrix).ravel()
    mejores = np.argsort(-similitudes, kind="stable")[:n]
    resultado = df.iloc[mejores][["recipe_id", "name", "country", "difficulty", "total_time_min"]].copy()
    resultado["similitud"] = similitudes[mejores].round(3)
    return resultado.reset_index(drop=True)


recomendar_por_texto("pollo jitomate chile")

In [ ]:
recomendar_por_texto("chocolate fresa crema")

## 3. Guardar y comprobar
Se guardan el vectorizador (convierte texto nuevo en números con el mismo vocabulario) y la matriz. Después se cargan con la misma función que usa el backend y se comprueba que corresponden al CSV.

In [ ]:
from src.models import load_artifacts, validate_alignment

MODELS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(tfidf, MODELS_DIR / "tfidf_vectorizer.joblib")
joblib.dump(tfidf_matrix, MODELS_DIR / "recipe_tfidf_matrix.joblib")

validate_alignment(load_artifacts(MODELS_DIR), df["recipe_text"].tolist())     # lanza error si algo no cuadra
for archivo in sorted(MODELS_DIR.glob("*.joblib")):
    print(f"{archivo.name:30} {archivo.stat().st_size / 1e6:5.1f} MB")